In [1]:
import pandas as pd
import numpy as np

In [2]:
popylation = pd.read_excel('data/Demography.xls', sheet_name="1.1", skiprows=6, header=None)
avg_age = pd.read_excel('data/Demography.xls', sheet_name="1.5 ", skiprows=10, header=None)

In [3]:
def format_population_data(df):
    formatted_column_row = {value: idx for idx, value in enumerate(df.iloc[1:, 0])}
    formatted_header_row = {int(value): idx for idx, value in enumerate(df.iloc[0, 1:])}
    formatted_df = df.iloc[1:, 1:].apply(pd.to_numeric, errors='coerce').replace({np.nan: None})
    
    return formatted_column_row, formatted_header_row, formatted_df


In [4]:
class Cube:
    def __init__(self, population_formatted=None, name="Население"):
        if population_formatted is None:
            self.regions = []
            self.years = []
            self.indicators = []
            self.cube = np.zeros((0, 0, 0))
            self.region_to_idx = {}
            self.year_to_idx = {}
            self.indicator_to_idx = {}
            print("Создан пустой куб")
        else:
            self.regions = list(population_formatted[0].keys())
            self.years = list(population_formatted[1].keys())
            self.indicators = [name]
            
            self.cube = np.zeros((len(self.regions), len(self.years), len(self.indicators)))

            self.region_to_idx = {region: idx for idx, region in enumerate(self.regions)}
            self.year_to_idx = {year: idx for idx, year in enumerate(self.years)}
            self.indicator_to_idx = {indicator: idx for idx, indicator in enumerate(self.indicators)}

            data_df = population_formatted[2]
            row_labels = population_formatted[0]  
            col_labels = population_formatted[1]  
            
            for region, row_idx in row_labels.items():
                for year, col_idx in col_labels.items():
                    try:
                        value = data_df.iloc[row_idx, col_idx]
                        self.cube[row_idx, col_idx, 0] = value  
                    except (IndexError, KeyError, ValueError):
                        self.cube[row_idx, col_idx, 0] = np.nan
            
            print(f"Куб создан: {self.cube.shape}")

    def add_indicator(self, new_data_formatted, indicator_name):
        """Добавляет новый показатель в куб, автоматически дополняя регионы и годы"""
        if indicator_name in self.indicators:
            print(f"Показатель '{indicator_name}' уже есть в кубе")
            return

        new_row_labels, new_col_labels, new_data_df = new_data_formatted
        
        new_regions = list(new_row_labels.keys())
        new_years = list(new_col_labels.keys())
        
        all_regions = list(set(self.regions + new_regions))
        all_years = list(set(self.years + new_years))
        
        new_cube = np.full((len(all_regions), len(all_years), len(self.indicators) + 1), np.nan)
        
        new_region_to_idx = {region: idx for idx, region in enumerate(all_regions)}
        new_year_to_idx = {year: idx for idx, year in enumerate(all_years)}
        
        if self.cube.shape[2] > 0:  
            for old_region_idx, region in enumerate(self.regions):
                new_region_idx = new_region_to_idx[region]
                for old_year_idx, year in enumerate(self.years):
                    new_year_idx = new_year_to_idx[year]
                    new_cube[new_region_idx, new_year_idx, :self.cube.shape[2]] = \
                        self.cube[old_region_idx, old_year_idx, :]
        
        for region, new_row_idx in new_row_labels.items():
            new_region_idx = new_region_to_idx[region]
            for year, new_col_idx in new_col_labels.items():
                new_year_idx = new_year_to_idx[year]
                try:
                    value = new_data_df.iloc[new_row_idx, new_col_idx]
                    new_cube[new_region_idx, new_year_idx, -1] = value
                except (IndexError, KeyError, ValueError):
                    new_cube[new_region_idx, new_year_idx, -1] = np.nan

        self.regions = all_regions
        self.years = all_years
        self.region_to_idx = new_region_to_idx
        self.year_to_idx = new_year_to_idx
        self.cube = new_cube
        self.indicators.append(indicator_name)
        self.indicator_to_idx[indicator_name] = len(self.indicators) - 1
        
        print(f"Добавлен показатель '{indicator_name}'")
        print(f"Новый размер куба: {self.cube.shape}")
        print(f"Регионы: {len(self.regions)}, Годы: {len(self.years)}")
        
    
    def get_data(self, region, year, metric="Население"):
        """Получить данные для региона и года"""
        if region in self.region_to_idx and year in self.year_to_idx and metric in self.indicator_to_idx:
            metric_idx = self.indicator_to_idx[metric]
            return self.cube[self.region_to_idx[region], self.year_to_idx[year], metric_idx]
        return None
    
    def get_row_data(self, region, metric="Население"):
        """Получить все данные по региону за все годы"""
        if region in self.region_to_idx and metric in self.indicator_to_idx:
            return self.cube[self.region_to_idx[region], :, self.indicator_to_idx[metric]]
        return None
    
    def get_col_data(self, year, metric="Население"):
        """Получить данные по всем регионам за определенный год"""
        if year in self.year_to_idx and metric in self.indicator_to_idx:
            return self.cube[:, self.year_to_idx[year], self.indicator_to_idx[metric]]
        return None
    
    def get_table(self, metric="Население"):
        """Получить всю таблицу по показателю"""
        if metric in self.indicator_to_idx:
            return self.cube[:, :, self.indicator_to_idx[metric]]
        return None
    
    def get_indicator_data(self, indicator_name):
        """Возвращает данные по указанному показателю в виде 2D массива [регионы × годы]"""
        if indicator_name not in self.indicator_to_idx:
            print(f"Показатель '{indicator_name}' не найден в кубе")
            return None
        
        indicator_idx = self.indicator_to_idx[indicator_name]
        return self.cube[:, :, indicator_idx].copy()

    def info(self):
        """Вывести информацию о кубе"""
        print(f"Размер куба: {self.cube.shape}")
        print(f"Регионы: {len(self.regions)}")
        print(f"Годы: {len(self.years)}")
        print(f"Показатели: {self.indicators}")

class BaseIndicator(Cube):
    """Класс для работы с базовыми показателями и их преобразованием"""
    
    def __init__(self, population_formatted=None, name="Население"):
        super().__init__(population_formatted, name)
        
    def fill_missing_values(self, indicator_name, method='zero', custom_value=None):
        """Заменяет пропущенные значения (NaN/None) в указанном показателе"""
        if indicator_name not in self.indicator_to_idx:
            print(f"Показатель '{indicator_name}' не найден в кубе")
            return
        
        indicator_idx = self.indicator_to_idx[indicator_name]
        data = self.cube[:, :, indicator_idx]

        mask = np.isnan(data)
        
        if not np.any(mask):
            print(f"В показателе '{indicator_name}' нет пропущенных значений")
            return

        if method == 'zero':
            fill_value = 0
        elif method == 'mean':
            # Среднее значение по не-NaN данным
            fill_value = np.nanmean(data)
        elif method == 'median':
            # Медиана по не-NaN данным
            fill_value = np.nanmedian(data)
        elif method == 'min':
            # Минимальное значение по не-NaN данным
            fill_value = np.nanmin(data)
        elif method == 'max':
            # Максимальное значение по не-NaN данным
            fill_value = np.nanmax(data)
        elif method == 'custom' and custom_value is not None:
            fill_value = custom_value
        else:
            print(f"Метод '{method}' не поддерживается. Доступные: zero, mean, median, min, max, forward_fill, backward_fill, custom")
            return

        filled_data = data.copy()
        filled_data[mask] = fill_value
        self.cube[:, :, indicator_idx] = filled_data
        
        print(f"Заполнено {np.sum(mask)} пропущенных значений в '{indicator_name}' методом {method} (значение: {fill_value})")

    def create_custom_indicator(self, formula, new_indicator_name):
        """Создает новый показатель по пользовательской формуле"""
        if new_indicator_name in self.indicators:
            print(f"Показатель '{new_indicator_name}' уже существует")
            return

        import re
        indicators_in_formula = re.findall(r'"([^"]+)"', formula)
        
        for indicator in indicators_in_formula:
            if indicator not in self.indicator_to_idx:
                print(f"Показатель '{indicator}' не найден в кубе")
                return

        new_cube = np.zeros((self.cube.shape[0], self.cube.shape[1], self.cube.shape[2] + 1))
        new_cube[:, :, :self.cube.shape[2]] = self.cube
        
        python_formula = formula
        for indicator in indicators_in_formula:
            idx = self.indicator_to_idx[indicator]
            python_formula = python_formula.replace(f'"{indicator}"', f'new_cube[:, :, {idx}]')

        try:
            result = eval(python_formula)
            result = np.where(np.isinf(result), np.nan, result)
            result = np.where(np.isnan(result), np.nan, result)
            new_cube[:, :, -1] = result
        except Exception as e:
            print(f"Ошибка при вычислении формулы: {e}")
            return

        self.cube = new_cube
        self.indicators.append(new_indicator_name)
        self.indicator_to_idx[new_indicator_name] = len(self.indicators) - 1
        
        print(f"Создан показатель '{new_indicator_name}' по формуле: {formula}")

In [5]:
class NormalizedIndicator(Cube):    
    def __init__(self, source_cube):
        super().__init__(None)
        
        self.source_cube = source_cube
        self.regions = source_cube.regions.copy()
        self.years = source_cube.years.copy()
        self.region_to_idx = source_cube.region_to_idx.copy()
        self.year_to_idx = source_cube.year_to_idx.copy()
        
        self.cube = np.zeros((len(self.regions), len(self.years), 0))
        self.indicators = []
        self.indicator_to_idx = {}
        
        print(f"Создан NormalizedIndicator для куба: {source_cube.cube.shape}")
        
    def create_normalized_indicator(self, indicator_name, method='minmax', new_indicator_name=None):
        """Создает нормализованную версию указанного показателя и добавляет в свой куб"""
        if indicator_name not in self.source_cube.indicator_to_idx:
            print(f"Показатель '{indicator_name}' не найден в исходном кубе")
            return
        
        if new_indicator_name is None:
            new_indicator_name = f"{indicator_name} ({method})"
        
        if new_indicator_name in self.indicators:
            print(f"Нормализованный показатель '{new_indicator_name}' уже существует")
            return

        data = self.source_cube.get_indicator_data(indicator_name)

        if data is None:
            print(f"Не удалось получить данные из показателя {indicator_name}")
            return
        
        if method == 'z-score':
            # Z-нормализация: (x - mean) / std
            mean = np.nanmean(data)
            std = np.nanstd(data)
            if std == 0:
                print("Стандартное отклонение равно 0, нормализация невозможна")
                return
            normalized_data = (data - mean) / std
            
        elif method == 'minmax':
            # Min-Max нормализация: (x - min) / (max - min)
            min_val = np.nanmin(data)
            max_val = np.nanmax(data)
            if max_val == min_val:
                print("Минимум и максимум равны, нормализация невозможна")
                return
            normalized_data = (data - min_val) / (max_val - min_val)
            
        elif method == 'robust':
            # Robust нормализация: (x - median) / IQR
            median = np.nanmedian(data)
            q75, q25 = np.nanpercentile(data, [75, 25])
            iqr = q75 - q25
            if iqr == 0:
                print("IQR равно 0, нормализация невозможна")
                return
            normalized_data = (data - median) / iqr
            
        else:
            print(f"Метод '{method}' не поддерживается. Доступные: z-score, minmax, robust")
            return
        self._add_normalized_data(normalized_data, new_indicator_name)
        print(f"Создан нормализованный показатель '{new_indicator_name}' методом {method}")

    def _add_normalized_data(self, normalized_data, indicator_name):
        """Внутренний метод для добавления нормализованных данных в куб"""
        new_cube = np.zeros((self.cube.shape[0], self.cube.shape[1], self.cube.shape[2] + 1))
        if self.cube.shape[2] > 0:
            new_cube[:, :, :self.cube.shape[2]] = self.cube
        new_cube[:, :, -1] = normalized_data
        self.cube = new_cube
        self.indicators.append(indicator_name)
        self.indicator_to_idx[indicator_name] = len(self.indicators) - 1

    def add_indicator(self, new_data_formatted, indicator_name):
        """Переопределяем метод добавления показателя для нормализованного куба"""
        print("Для NormalizedIndicator используйте create_normalized_indicator()")
        return

    def copy_normalized_to_source(self, normalized_indicator_name, target_indicator_name=None):
        """Копирует нормализованный показатель обратно в исходный куб"""
        if normalized_indicator_name not in self.indicator_to_idx:
            print(f"Нормализованный показатель '{normalized_indicator_name}' не найден")
            return
        
        if target_indicator_name is None:
            target_indicator_name = normalized_indicator_name
        
        # Создаем фиктивный formatted_data для добавления в исходный куб
        dummy_formatted = (
            {region: idx for idx, region in enumerate(self.regions)},
            {year: idx for idx, year in enumerate(self.years)},
            pd.DataFrame(self.get_table(normalized_indicator_name))
        )
        
        self.source_cube.add_indicator(dummy_formatted, target_indicator_name)
        print(f"Нормализованный показатель '{normalized_indicator_name}' скопирован в исходный куб как '{target_indicator_name}'")

In [6]:
WeightsIndicator = pd.read_excel('data/Weights.xlsx', header=None)

def parse_weights_data(df):
    weights_dict = {}
    for idx in range(len(df)):
        try:
            indicator_name = df.iloc[idx, 0]
            weight_value = df.iloc[idx, 1]
            
            if pd.notna(indicator_name) and pd.notna(weight_value):
                weights_dict[str(indicator_name).strip()] = float(weight_value)
                
        except (ValueError, TypeError, IndexError) as e:
            print(f"Ошибка при обработке строки {idx}: {e}")
            continue
    
    return weights_dict


In [7]:
class WeightedIndicator:
    def __init__(self, weights_dict):
        self.weights_dict = weights_dict #{название_показателя: вес}
        self.weighted_indicators = {}
    
    def apply_weights(self, cube, year, name='Взвешенный показатель'):
        indicator_names = list(self.weights_dict.keys())
        
        if not indicator_names:
            print("В словаре весов нет показателей")
            return None
        
        missing_indicators = []
        for indicator in indicator_names:
            if indicator not in cube.indicators:
                missing_indicators.append(indicator)
        
        if missing_indicators:
            print(f"Показатели не найдены в кубе: {missing_indicators}")
            print(f"Доступные показатели в кубе: {cube.indicators}")
            return None
        
        if year not in cube.year_to_idx:
            print(f"Год {year} не найден в кубе")
            print(f"Доступные годы: {list(cube.year_to_idx.keys())}")
            return None
        
        weighted_table = np.zeros((len(indicator_names), len(cube.regions)))
        
        print(f"Создаем таблицу размером: {weighted_table.shape} (показатели × регионы)")
        
        for i, indicator in enumerate(indicator_names):
            year_data = cube.get_col_data(year, indicator)
            weight = self.weights_dict[indicator]
            
            weighted_data = year_data * weight
            weighted_table[i, :] = weighted_data
            
        
        self.weighted_indicators[name] = {
            'data': weighted_table,
            'indicators': indicator_names,
            'regions': cube.regions,
            'year': year,
            'weights': [self.weights_dict[ind] for ind in indicator_names]
        }
        
        print(f"Создана таблица взвешенных показателей '{name}'")
        print(f"Размер таблицы: {weighted_table.shape}")
        print(f"Показатели (строки): {indicator_names}")
        print(f"Регионы (столбцы): {len(cube.regions)} регионов")
        print(f"Веса: {[self.weights_dict[ind] for ind in indicator_names]}")
        
        return weighted_table

    def get_weighted_table(self, weighted_indicator_name):
        """Получить всю таблицу взвешенных данных по показателю"""
        if weighted_indicator_name not in self.weighted_indicators:
            print(f"Взвешенный показатель '{weighted_indicator_name}' не найден")
            print(f"Доступные взвешенные показатели: {list(self.weighted_indicators.keys())}")
            return None
        
        return self.weighted_indicators[weighted_indicator_name]['data']
    
    def get_weighted_dataframe(self, weighted_indicator_name):
        """Получить таблицу в виде DataFrame с названиями показателей и регионов"""
        if weighted_indicator_name not in self.weighted_indicators:
            print(f"Взвешенный показатель '{weighted_indicator_name}' не найден")
            return None
        
        data = self.weighted_indicators[weighted_indicator_name]
        df = pd.DataFrame(
            data['data'],
            index=data['indicators'],  # строки - показатели
            columns=data['regions']    # столбцы - регионы
        )
        return df
    
    def get_region_column(self, weighted_indicator_name, region):
        if weighted_indicator_name not in self.weighted_indicators:
            print(f"Взвешенный показатель '{weighted_indicator_name}' не найден")
            return None
        
        data = self.weighted_indicators[weighted_indicator_name]
        if region not in data['regions']:
            print(f"Регион '{region}' не найден в таблице")
            return None
        
        idx = data['regions'].index(region)
        return data['data'][:, idx]
    
    def info(self):
        print("Взвешенные таблицы:")
        for name, data in self.weighted_indicators.items():
            print(f"  '{name}': {data['data'].shape} (показатели × регионы), год: {data['year']}")
            print(f"    Показатели: {data['indicators']}")

In [8]:
population_formatted = format_population_data(popylation)
cube = BaseIndicator()
cube.add_indicator(population_formatted, 'Население')
avg_age_formatted = format_population_data(popylation)
cube.add_indicator(avg_age_formatted, "Средний возраст")

cube.info()
cube.create_custom_indicator('"Население"*"Средний возраст"', "Базовый показатель")
cube.get_data("Российская Федерация", 2000, "Демографический индекс")

# cube.create_normalized_indicator("Население")
# cube.fill_missing_values("Население", "mean")
# cube.create_normalized_indicator("Население", method='robust')

Создан пустой куб
Добавлен показатель 'Население'
Новый размер куба: (97, 16, 1)
Регионы: 97, Годы: 16
Добавлен показатель 'Средний возраст'
Новый размер куба: (97, 16, 2)
Регионы: 97, Годы: 16
Размер куба: (97, 16, 2)
Регионы: 97
Годы: 16
Показатели: ['Население', 'Средний возраст']
Создан показатель 'Базовый показатель' по формуле: "Население"*"Средний возраст"


In [9]:
norm = NormalizedIndicator(cube)
print(cube.get_table('Базовый показатель'))
norm.create_normalized_indicator("Базовый показатель", method='minmax', new_indicator_name="Базовый показатель")
print(norm.get_table("Базовый показатель"))

Создан пустой куб
Создан NormalizedIndicator для куба: (97, 16, 3)
[[1075369.   1097046.76 1103550.25 ... 1033678.89 1042236.81 1065230.41]
 [  94126.24   92112.25   91506.25 ...  100362.24   98784.49   95976.04]
 [   1806.25    1789.29    1764.   ...    1789.29    1780.84    1789.29]
 ...
 [ 679140.81  658532.25  633297.64 ...  752035.84  726074.41  702411.61]
 [  21170.25   20909.16   20420.41 ...   23104.     22440.04   21726.76]
 [ 290413.21  291708.01  292140.25 ...  284835.69  286439.04  288691.29]]
Создан нормализованный показатель 'Базовый показатель' методом minmax
[[4.90441201e-05 5.00343360e-05 5.03314082e-05 ... 4.71397623e-05
  4.75306786e-05 4.85810006e-05]
 [4.22204456e-06 4.13004774e-06 4.10236633e-06 ... 4.50689809e-06
  4.43482823e-06 4.30654136e-06]
 [4.97031951e-09 4.19560562e-09 3.04038661e-09 ... 4.19560562e-09
  3.80961904e-09 4.19560562e-09]
 ...
 [3.09448580e-05 3.00034820e-05 2.88507931e-05 ... 3.42746217e-05
  3.30887326e-05 3.20078422e-05]
 [8.89496255e-07 8

In [ ]:
weights_data = {'Население': 2, 'Средний возраст': 160.0}
weighted_calc = WeightedIndicator(weights_data)

result = weighted_calc.apply_weights(
    cube=cube,
    year=2005,
    name='Взвешенный'
)
print(weighted_calc.get_weighted_table('Взвешенный'))


Создаем таблицу размером: (2, 97) (показатели × регионы)
Создана таблица взвешенных показателей 'Взвешенный'
Размер таблицы: (2, 97)
Показатели (строки): ['Население', 'Средний возраст']
Регионы (столбцы): 97 регионов
Веса: [2, 160.0]
